# Evaluation Reproducibility Notebook

This notebook verifies that the evaluation workflow can reproduce the reported metrics when run with the saved checkpoint.

It performs the following steps:

1. Checks that the saved model checkpoint exists.
2. Generates test predictions from the checkpoint.
3. Runs the evaluation script.
4. Verifies that metrics and figures are saved.
5. Displays the reproduced evaluation metrics.

In [ ]:
from pathlib import Path

checkpoint_path = Path("../artifacts/checkpoints/best_model.pt")
test_data_dir = Path("../data/raw/chest_xray/test")

predictions_path = Path("../artifacts/predictions/test_predictions.csv")
metrics_path = Path("../artifacts/reports/evaluation_metrics.json")
confusion_matrix_path = Path("../artifacts/figures/confusion_matrix_test.png")
roc_curve_path = Path("../artifacts/figures/roc_curve.png")

paths = {
    "checkpoint": checkpoint_path,
    "test_data_dir": test_data_dir,
}

for name, path in paths.items():
    print(f"{name}: {path} -> exists={path.exists()}")

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "-m",
    "src.evaluation.generate_test_predictions",
    "--checkpoint",
    "artifacts/checkpoints/best_model.pt",
    "--data-dir",
    "data/raw/chest_xray/test",
    "--output-csv",
    "artifacts/predictions/test_predictions.csv",
]

result = subprocess.run(cmd, cwd="..", capture_output=True, text=True)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Test prediction generation failed.")

In [ ]:
cmd = [
    sys.executable,
    "-m",
    "src.evaluation.evaluate",
    "--input-csv",
    "artifacts/predictions/test_predictions.csv",
    "--threshold",
    "0.5",
    "--metrics-out",
    "artifacts/reports/evaluation_metrics.json",
    "--confusion-matrix-out",
    "artifacts/figures/confusion_matrix_test.png",
    "--roc-curve-out",
    "artifacts/figures/roc_curve.png",
]

result = subprocess.run(cmd, cwd="..", capture_output=True, text=True)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Evaluation failed.")

In [ ]:
outputs = {
    "predictions_csv": predictions_path,
    "metrics_json": metrics_path,
    "confusion_matrix": confusion_matrix_path,
    "roc_curve": roc_curve_path,
}

for name, path in outputs.items():
    print(f"{name}: {path} -> exists={path.exists()}")

In [ ]:
import json
import pandas as pd

with metrics_path.open("r", encoding="utf-8") as f:
    metrics = json.load(f)

summary = {
    "threshold": metrics.get("threshold"),
    "accuracy": metrics.get("accuracy"),
    "precision": metrics.get("precision"),
    "sensitivity": metrics.get("sensitivity"),
    "specificity": metrics.get("specificity"),
    "roc_auc": metrics.get("roc_auc"),
}

pd.DataFrame([summary])